In [1]:
import numpy as np
import pandas as pd
import difflib

# Task 1

In [2]:
bikes = pd.read_csv('bike_listings_s7.csv')

exact_copies = bikes.sample(25, random_state=42)

relisted = bikes.sample(10, random_state=1).copy()
relisted['price_inr'] = relisted['price_inr'] * 0.95

bikes_dup = pd.concat([bikes, exact_copies, relisted], ignore_index=True)
print("Rows after adding duplicates:", len(bikes_dup))

print("Exact copies:", bikes_dup.duplicated().sum())

same_id = bikes_dup[bikes_dup.duplicated(subset='listing_id', keep=False)].sort_values('listing_id')
print("Rows sharing a listing_id:", len(same_id))
print(same_id.head(10))

Rows after adding duplicates: 435
Exact copies: 25
Rows sharing a listing_id: 70
     listing_id   brand       city  model_year  km_driven  owner_count  \
342           5  Sundar     Rajkot        2021    81252.0            3   
430           5  Sundar     Rajkot        2021    81252.0            3   
408          11  Sundar   Vadodara        2018    25624.0            2   
266          11  Sundar   Vadodara        2018    25624.0            2   
425          25  Tarang     Rajkot        2023    81714.0            2   
398          25  Tarang     Rajkot        2023    81714.0            2   
126          32  Veloce      Surat        2012    30699.0            2   
409          32  Veloce      Surat        2012    30699.0            2   
385          85  Sundar  Ahmedabad        2023   400000.0            1   
419          85  Sundar  Ahmedabad        2023   400000.0            1   

     price_inr  price_capped  price_wins     km_log  is_first_owner  \
342   27469.00       27469.0    3

# Task 2

In [3]:
bikes_nodup = bikes_dup.drop_duplicates()
print("After drop_duplicates():", len(bikes_nodup))

bikes_unique = bikes_nodup.drop_duplicates(subset='listing_id', keep='last')
print("After keeping latest per listing_id:", len(bikes_unique))

assert len(bikes_unique) == 400
assert bikes_unique['listing_id'].is_unique

After drop_duplicates(): 410
After keeping latest per listing_id: 400


# Task 3

In [4]:
np.random.seed(42)
messy = bikes_unique.copy()

veloce_rows = np.random.choice(messy[messy['brand'] == 'Veloce'].index, 12, replace=False)
messy.loc[veloce_rows, 'brand'] = 'Velose'
ahmedabad_rows = np.random.choice(messy[messy['city'] == 'Ahmedabad'].index, 12, replace=False)
messy.loc[ahmedabad_rows, 'city'] = 'Ahmdabad'

lower_rows = np.random.choice(messy.index, 30, replace=False)
messy.loc[lower_rows, 'brand'] = messy.loc[lower_rows, 'brand'].str.lower()
messy.loc[lower_rows, 'city'] = messy.loc[lower_rows, 'city'].str.lower()

space_rows = np.random.choice(messy.index, 20, replace=False)
messy.loc[space_rows, 'brand'] = messy.loc[space_rows, 'brand'] + ' '
messy.loc[space_rows, 'city'] = messy.loc[space_rows, 'city'] + ' '

print("Before fixing:")
print(messy['brand'].value_counts())
print(messy['city'].value_counts())

for col in ['brand', 'city']:
    messy[col] = messy[col].str.strip().str.title()

messy['brand'] = messy['brand'].replace({'Velose': 'Veloce'})
messy['city'] = messy['city'].replace({'Ahmdabad': 'Ahmedabad'})

valid_brands = ['Veloce', 'Rajhans', 'Sundar', 'Tarang']
valid_cities = ['Ahmedabad', 'Surat', 'Vadodara', 'Rajkot']

def fix_value(value, valid_list):
    match = difflib.get_close_matches(value, valid_list, n=1, cutoff=0.8)
    if match:
        return match[0]
    return value

messy['brand'] = messy['brand'].apply(fix_value, valid_list=valid_brands)
messy['city'] = messy['city'].apply(fix_value, valid_list=valid_cities)

print("After fixing:")
print(messy['brand'].value_counts())
print(messy['city'].value_counts())
assert messy['brand'].nunique() == 4
assert messy['city'].nunique() == 4

Before fixing:
brand
Tarang      107
Sundar       88
Rajhans      75
Veloce       70
Velose       10
sundar        8
Sundar        8
veloce        8
rajhans       7
tarang        6
Veloce        5
Rajhans       4
Tarang        2
Velose        1
velose        1
Name: count, dtype: int64
city
Rajkot        96
Surat         85
Ahmedabad     81
Vadodara      76
Ahmdabad      12
rajkot        12
Surat         10
surat          6
ahmedabad      6
vadodara       6
Ahmedabad      5
Rajkot         4
Vadodara       1
Name: count, dtype: int64
After fixing:
brand
Tarang     115
Sundar     104
Veloce      95
Rajhans     86
Name: count, dtype: int64
city
Rajkot       112
Ahmedabad    104
Surat        101
Vadodara      83
Name: count, dtype: int64


# Task 4

In [5]:
repaired = messy.copy()

repaired['is_insured'] = np.random.choice(['Yes', 'yes', 'Y', 'NO', 'no', 'N', ' yes '], len(repaired))
print(repaired['is_insured'].value_counts())

cleaned = repaired['is_insured'].str.strip().str.lower()
repaired['is_insured'] = cleaned.map({'yes': 1, 'y': 1, 'no': 0, 'n': 0})

assert repaired['listing_id'].nunique() == 400
assert repaired['brand'].nunique() == 4
assert repaired['city'].nunique() == 4
assert set(repaired['is_insured'].unique()) <= {0, 1}

print(repaired['is_insured'].value_counts())
repaired.to_csv('bike_listings_clean.csv', index=False)

is_insured
 yes     63
N        61
no       59
Yes      58
Y        56
yes      55
NO       48
Name: count, dtype: int64
is_insured
1    232
0    168
Name: count, dtype: int64
